# Notebook 7: 06_business_conclusion
Portfolio-style business conclusion output.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
AUDIO_PATH = DATA_DIR / 'audio_clip_event_cleaned.csv'
if not AUDIO_PATH.exists():
    AUDIO_PATH = DATA_DIR / 'audio_clip_event.csv'
THR_LOG = DATA_DIR / 'threshold_experiment_log.csv'

df = pd.read_csv(AUDIO_PATH)
thr_log = pd.read_csv(THR_LOG)
print('Using event file:', AUDIO_PATH)


In [ ]:
def metrics(df, thr):
    pred = (df['model_score'] >= thr).astype(int)
    y = df['binary_label'].astype(int)
    tp = int(((y==1)&(pred==1)).sum())
    fp = int(((y==0)&(pred==1)).sum())
    fn = int(((y==1)&(pred==0)).sum())
    precision = tp/(tp+fp) if (tp+fp) else 0.0
    recall = tp/(tp+fn) if (tp+fn) else 0.0
    return tp, fp, fn, precision, recall

n_pos = int((df['binary_label']==1).sum())
n_neg = int((df['binary_label']==0).sum())
fp_root = df[df['error_type']=='FP']['true_class'].value_counts().head(3)
scene_fp = df.groupby('scene_type').apply(lambda x: (x['error_type']=='FP').sum()/len(x)).sort_values(ascending=False)
time_fp = df.groupby('time_bucket').apply(lambda x: (x['error_type']=='FP').sum()/len(x)).sort_values(ascending=False)

tp80, fp80, fn80, p80, r80 = metrics(df, 0.80)
tp86, fp86, fn86, p86, r86 = metrics(df, 0.86)

print('Positive:', n_pos, 'Negative:', n_neg)
print('Top FP source:')
print(fp_root)
print('Highest FP scene:', scene_fp.index[0], round(float(scene_fp.iloc[0]), 4))
print('Highest FP time bucket:', time_fp.index[0], round(float(time_fp.iloc[0]), 4))
print('Threshold 0.80 -> precision={:.4f}, recall={:.4f}, FP={}, FN={}'.format(p80,r80,fp80,fn80))
print('Threshold 0.86 -> precision={:.4f}, recall={:.4f}, FP={}, FN={}'.format(p86,r86,fp86,fn86))


In [ ]:
conclusion_text = (
    f"Business conclusion draft:\n\n"
    f"1. The dataset is clearly imbalanced, with negative samples ({n_neg}) much higher than positive samples ({n_pos}).\n"
    f"2. False alarms are concentrated in high-energy sudden noises, especially {fp_root.index[0]} and {fp_root.index[1]}.\n"
    f"3. FP risk is highest in scene '{scene_fp.index[0]}' and time bucket '{time_fp.index[0]}'.\n"
    f"4. Score overlap in medium-high confidence indicates hard boundaries between fireworks and similar noise classes.\n"
    f"5. Precision increases from {p80:.3f} (threshold 0.80) to {p86:.3f} (threshold 0.86), supporting a precision-priority deployment strategy.\n"
    f"6. Recommended action: deploy around threshold 0.85-0.86 and monitor FN risk through sampled manual audits."
)
print(conclusion_text)
